In [1]:
# ============================================================
# FEATURE ENGINEERING
# Proyecto: Optimización de rutas de recolección de residuos
# Municipalidad Distrital de Yarinacocha - 2026
# ============================================================

# 1. LIBRERÍAS
import pandas as pd
import numpy as np
import re

from google.colab import files
from sklearn.preprocessing import StandardScaler

print("=== FEATURE ENGINEERING - RESIDUOS YARINACOCHA 2026 ===")


# ============================================================
# 2. CARGAR ARCHIVO CSV
# ============================================================

print("\nSube el archivo: residuos_yarinacocha_2026.csv")

uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]

df = pd.read_csv(nombre_archivo)

print("\nArchivo cargado correctamente:", nombre_archivo)
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])


# ============================================================
# 3. MOSTRAR DATOS INICIALES
# ============================================================

print("\n--- Primeras filas ---")
display(df.head())

print("\n--- Información del dataset ---")
df.info()


# ============================================================
# 4. NORMALIZAR NOMBRES DE COLUMNAS
# ============================================================

def limpiar_nombre(texto):
    texto = str(texto).strip().lower()

    reemplazos = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ñ": "n"
    }

    for original, nuevo in reemplazos.items():
        texto = texto.replace(original, nuevo)

    texto = re.sub(r'[^a-zA-Z0-9]+', '_', texto)
    texto = texto.strip('_')

    return texto


df.columns = [limpiar_nombre(c) for c in df.columns]

print("\n--- Nombres de columnas normalizados ---")
print(df.columns.tolist())


# ============================================================
# 5. ELIMINAR REGISTROS DUPLICADOS
# ============================================================

duplicados = df.duplicated().sum()

print("\nRegistros duplicados encontrados:", duplicados)

df = df.drop_duplicates().reset_index(drop=True)

print("Registros después de eliminar duplicados:", len(df))


# ============================================================
# 6. VALORES FALTANTES
# ============================================================

print("\n--- Valores faltantes antes del tratamiento ---")

faltantes = df.isnull().sum()
print(faltantes[faltantes > 0])

# Variables numéricas: reemplazar por mediana
columnas_numericas = df.select_dtypes(include=np.number).columns

for columna in columnas_numericas:

    if df[columna].isnull().sum() > 0:

        df[columna] = df[columna].fillna(
            df[columna].median()
        )

# Variables categóricas: reemplazar por moda
columnas_categoricas = df.select_dtypes(
    include=['object', 'category']
).columns

for columna in columnas_categoricas:

    if df[columna].isnull().sum() > 0:

        moda = df[columna].mode()

        if len(moda) > 0:
            df[columna] = df[columna].fillna(moda[0])

print("\nValores faltantes después del tratamiento:")
print(df.isnull().sum().sum())


# ============================================================
# 7. FUNCIÓN PARA IDENTIFICAR COLUMNAS
# ============================================================

def buscar_columna(posibles_nombres):

    for posible in posibles_nombres:

        for columna in df.columns:

            if posible in columna:
                return columna

    return None


# Buscar automáticamente las principales variables

col_distancia = buscar_columna([
    'distancia_km',
    'distancia',
    'kilometros',
    'km_recorridos'
])

col_tiempo = buscar_columna([
    'tiempo_min',
    'tiempo',
    'duracion',
    'minutos'
])

col_combustible = buscar_columna([
    'combustible_litros',
    'combustible',
    'litros',
    'consumo_combustible'
])

col_costo = buscar_columna([
    'costo_operativo',
    'costo',
    'coste'
])

col_puntos = buscar_columna([
    'puntos_atendidos',
    'puntos',
    'puntos_recoleccion'
])

col_peso = buscar_columna([
    'peso_recolectado',
    'peso_kg',
    'peso',
    'residuos_kg'
])

col_capacidad = buscar_columna([
    'capacidad_vehiculo',
    'capacidad_kg',
    'capacidad'
])


print("\n--- Variables identificadas ---")

print("Distancia:", col_distancia)
print("Tiempo:", col_tiempo)
print("Combustible:", col_combustible)
print("Costo:", col_costo)
print("Puntos atendidos:", col_puntos)
print("Peso recolectado:", col_peso)
print("Capacidad:", col_capacidad)


# ============================================================
# 8. CREACIÓN DE NUEVAS CARACTERÍSTICAS
# ============================================================

print("\n--- Creando nuevas características ---")


# ------------------------------------------------------------
# Distancia promedio por punto atendido
# ------------------------------------------------------------

if col_distancia and col_puntos:

    df['distancia_por_punto'] = np.where(
        df[col_puntos] > 0,
        df[col_distancia] / df[col_puntos],
        0
    )

    print("Creada: distancia_por_punto")


# ------------------------------------------------------------
# Tiempo promedio por punto atendido
# ------------------------------------------------------------

if col_tiempo and col_puntos:

    df['tiempo_por_punto'] = np.where(
        df[col_puntos] > 0,
        df[col_tiempo] / df[col_puntos],
        0
    )

    print("Creada: tiempo_por_punto")


# ------------------------------------------------------------
# Consumo de combustible por kilómetro
# ------------------------------------------------------------

if col_combustible and col_distancia:

    df['combustible_por_km'] = np.where(
        df[col_distancia] > 0,
        df[col_combustible] / df[col_distancia],
        0
    )

    print("Creada: combustible_por_km")


# ------------------------------------------------------------
# Costo operativo por kilómetro
# ------------------------------------------------------------

if col_costo and col_distancia:

    df['costo_por_km'] = np.where(
        df[col_distancia] > 0,
        df[col_costo] / df[col_distancia],
        0
    )

    print("Creada: costo_por_km")


# ------------------------------------------------------------
# Costo por punto atendido
# ------------------------------------------------------------

if col_costo and col_puntos:

    df['costo_por_punto'] = np.where(
        df[col_puntos] > 0,
        df[col_costo] / df[col_puntos],
        0
    )

    print("Creada: costo_por_punto")


# ------------------------------------------------------------
# Peso promedio recolectado por punto
# ------------------------------------------------------------

if col_peso and col_puntos:

    df['peso_por_punto'] = np.where(
        df[col_puntos] > 0,
        df[col_peso] / df[col_puntos],
        0
    )

    print("Creada: peso_por_punto")


# ------------------------------------------------------------
# Porcentaje de utilización de capacidad
# ------------------------------------------------------------

if col_peso and col_capacidad:

    df['porcentaje_capacidad'] = np.where(
        df[col_capacidad] > 0,
        (df[col_peso] / df[col_capacidad]) * 100,
        0
    )

    print("Creada: porcentaje_capacidad")


# ============================================================
# 9. PROCESAMIENTO DE FECHAS
# ============================================================

columnas_fecha = [
    c for c in df.columns
    if 'fecha' in c
]

for columna in columnas_fecha:

    try:

        df[columna] = pd.to_datetime(
            df[columna],
            errors='coerce',
            dayfirst=True
        )

        df[columna + '_anio'] = df[columna].dt.year
        df[columna + '_mes'] = df[columna].dt.month
        df[columna + '_dia'] = df[columna].dt.day
        df[columna + '_dia_semana'] = df[columna].dt.dayofweek

        print("Procesada fecha:", columna)

    except:
        pass


# ============================================================
# 10. TRATAMIENTO DE VALORES INFINITOS
# ============================================================

df.replace([np.inf, -np.inf], np.nan, inplace=True)

for columna in df.select_dtypes(include=np.number).columns:

    if df[columna].isnull().sum() > 0:

        df[columna] = df[columna].fillna(
            df[columna].median()
        )


# ============================================================
# 11. CODIFICACIÓN DE VARIABLES CATEGÓRICAS
# ============================================================

df_modelo = df.copy()

categoricas = df_modelo.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print("\nVariables categóricas encontradas:")
print(categoricas)

if len(categoricas) > 0:

    df_modelo = pd.get_dummies(
        df_modelo,
        columns=categoricas,
        drop_first=True,
        dtype=int
    )


# ============================================================
# 12. ELIMINAR COLUMNAS DE FECHA DEL DATASET DEL MODELO
# ============================================================

columnas_datetime = df_modelo.select_dtypes(
    include=['datetime64']
).columns

df_modelo = df_modelo.drop(
    columns=columnas_datetime,
    errors='ignore'
)


# ============================================================
# 13. ESCALAMIENTO DE VARIABLES NUMÉRICAS
# ============================================================

columnas_numericas_modelo = df_modelo.select_dtypes(
    include=np.number
).columns

df_escalado = df_modelo.copy()

if len(columnas_numericas_modelo) > 0:

    scaler = StandardScaler()

    df_escalado[columnas_numericas_modelo] = scaler.fit_transform(
        df_modelo[columnas_numericas_modelo]
    )


print("\nEscalamiento realizado correctamente.")


# ============================================================
# 14. RESUMEN DE LAS NUEVAS VARIABLES
# ============================================================

print("\n==========================================")
print("RESUMEN DEL FEATURE ENGINEERING")
print("==========================================")

print("Filas finales:", df.shape[0])
print("Columnas originales/procesadas:", df.shape[1])
print("Columnas después de codificación:", df_modelo.shape[1])

print("\nColumnas generadas:")

nuevas_variables = [
    'distancia_por_punto',
    'tiempo_por_punto',
    'combustible_por_km',
    'costo_por_km',
    'costo_por_punto',
    'peso_por_punto',
    'porcentaje_capacidad'
]

for variable in nuevas_variables:

    if variable in df.columns:
        print("✓", variable)


# ============================================================
# 15. ESTADÍSTICAS DESCRIPTIVAS
# ============================================================

print("\n--- Estadísticas de las variables ---")

display(
    df.select_dtypes(include=np.number)
      .describe()
      .T
)


# ============================================================
# 16. VISUALIZAR DATASET RESULTANTE
# ============================================================

print("\n--- Dataset después del Feature Engineering ---")

display(df.head(10))


# ============================================================
# 17. GUARDAR DATASET CON FEATURES
# ============================================================

archivo_features = "residuos_yarinacocha_feature_engineering.csv"

df.to_csv(
    archivo_features,
    index=False
)

print("\nArchivo generado:")
print(archivo_features)


# ============================================================
# 18. GUARDAR DATASET PREPARADO PARA MACHINE LEARNING
# ============================================================

archivo_modelo = "residuos_yarinacocha_ML.csv"

df_modelo.to_csv(
    archivo_modelo,
    index=False
)

print("\nArchivo generado:")
print(archivo_modelo)


# ============================================================
# 19. GUARDAR DATASET ESCALADO
# ============================================================

archivo_escalado = "residuos_yarinacocha_ML_escalado.csv"

df_escalado.to_csv(
    archivo_escalado,
    index=False
)

print("\nArchivo generado:")
print(archivo_escalado)


# ============================================================
# 20. DESCARGAR RESULTADOS
# ============================================================

print("\nProceso terminado correctamente.")
print("Puedes descargar los archivos generados.")

files.download(archivo_features)
files.download(archivo_modelo)
files.download(archivo_escalado)

=== FEATURE ENGINEERING - RESIDUOS YARINACOCHA 2026 ===

Sube el archivo: residuos_yarinacocha_2026.csv


Saving residuos_yarinacocha_2026.csv to residuos_yarinacocha_2026.csv

Archivo cargado correctamente: residuos_yarinacocha_2026.csv
Filas: 733
Columnas: 18

--- Primeras filas ---


,fecha,codigo_ruta,vehiculo,sector,hora_salida,hora_llegada,duracion_min,distancia_km,combustible_l,costo_operativo_soles,capacidad_vehicular_kg,residuos_recolectados_kg,puntos_programados,puntos_atendidos,cumplimiento_pct,latitud,longitud,incidencia
0,2026-01-01,R003,VH04,San José,06:20,09:46,206,41.95,13.57,348.39,6000,3503.32,40,40,100.0,-8.382415,-74.602529,Demora operativa
1,2026-01-01,R004,VH02,La Restinga,07:27,09:57,150,30.83,8.37,228.71,5000,3861.81,45,45,100.0,-8.314313,-74.544781,Ninguna
2,2026-01-02,R004,VH01,La Restinga,06:26,08:56,150,38.47,11.82,302.69,4500,3398.72,39,39,100.0,-8.363072,-74.596332,Ninguna
3,2026-01-02,R005,VH03,Las Palmeras,07:17,10:11,174,41.37,13.01,327.39,5500,4348.02,42,42,100.0,-8.351885,-74.573015,Ninguna
4,2026-01-03,R005,VH02,Las Palmeras,06:27,09:34,187,48.09,13.47,336.99,5000,4619.23,33,33,100.0,-8.369644,-74.602894,Ninguna



--- Información del dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 733 entries, 0 to 732
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   fecha                     733 non-null    object 
 1   codigo_ruta               733 non-null    object 
 2   vehiculo                  733 non-null    object 
 3   sector                    733 non-null    object 
 4   hora_salida               733 non-null    object 
 5   hora_llegada              733 non-null    object 
 6   duracion_min              733 non-null    int64  
 7   distancia_km              733 non-null    float64
 8   combustible_l             727 non-null    float64
 9   costo_operativo_soles     729 non-null    float64
 10  capacidad_vehicular_kg    733 non-null    int64  
 11  residuos_recolectados_kg  733 non-null    float64
 12  puntos_programados        733 non-null    int64  
 13  puntos_atendidos          733 no

,count,mean,std,min,25%,50%,75%,max
duracion_min,730.0,199.620548,44.346000,150.000000,160.000000,191.000000,225.750000,431.000000
distancia_km,730.0,40.296342,6.790340,21.760000,35.850000,39.820000,44.987500,60.270000
combustible_l,730.0,12.524507,2.371696,7.070000,10.970000,12.295000,14.017500,23.270000
costo_operativo_soles,730.0,329.289315,52.195331,216.400000,294.515000,323.315000,361.555000,554.390000
capacidad_vehicular_kg,730.0,5250.684932,559.399857,4500.000000,5000.000000,5250.000000,5875.000000,6000.000000
residuos_recolectados_kg,730.0,3833.594014,476.909568,2453.530000,3519.467500,3826.035000,4140.917500,5352.750000
puntos_programados,730.0,40.883562,5.203289,32.000000,36.000000,41.000000,45.000000,50.000000
puntos_atendidos,730.0,40.239726,5.275542,29.000000,36.000000,40.000000,45.000000,50.000000
cumplimiento_pct,730.0,98.399233,2.192214,90.620000,97.440000,100.000000,100.000000,100.000000
latitud,730.0,-8.353358,0.024689,-8.426376,-8.369878,-8.355445,-8.336553,-8.273710


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>